# 02 — Capa Silver: estandarización, calidad y cuarentena

## Objetivo

Este notebook sirve como guion técnico y visual para la exposición. Todas las
comprobaciones usan los artefactos completos del proyecto; las vistas de pocas filas
son únicamente para mostrar el esquema, no son el conjunto usado por el pipeline.

**QUÉ EXPLICAR:** primero diga qué problema resuelve esta etapa, después muestre la
evidencia y termine conectándola con la siguiente capa.

## Preparación

Ejecútelo desde el contenedor `spark` con la raíz `/workspace`. La celda también
encuentra el repositorio cuando el notebook se abre desde la carpeta local.

In [ ]:
from pathlib import Path
import json

# Encontrar la raíz permite usar el mismo notebook en Docker o en el repositorio local.
candidates = [Path.cwd(), Path.cwd().parent, Path('/workspace')]
ROOT = next(
    path for path in candidates
    if (path / 'config' / 'pipeline.yaml').exists()
)
DATA = ROOT / 'data'
EXPORTS = ROOT / 'exports'
print(f'Raíz del proyecto: {ROOT}')

## Pasos

### 1. Reconciliación completa

Silver unifica yellow, green, FHV y FHVHV. Cada registro fuente termina como válido o en
cuarentena; nunca se pierde silenciosamente.

**QUÉ EXPLICAR:** la igualdad clave es `fuente = válidas + cuarentena`.

In [ ]:
report = json.loads((EXPORTS / 'verification_report.json').read_text(encoding='utf-8'))
check = next(c for c in report['checks'] if c['name'] == 'silver_row_reconciliation')
d = check['details']
assert d['source_rows'] == d['valid_rows'] + d['quarantine_rows']
print(json.dumps(d, indent=2))
print('Diferencia:', d['source_rows'] - d['valid_rows'] - d['quarantine_rows'])

### 2. Esquema PySpark y muestra explicativa

In [ ]:
from pyspark.sql import SparkSession

# Solo se muestran 5 filas; PySpark lee el mismo Silver completo usado por Gold.
spark = (SparkSession.builder.master('local[*]')
         .appName('TLC-Exposicion-Silver').getOrCreate())
silver_df = spark.read.option('basePath', str(DATA / 'silver')).parquet(str(DATA / 'silver'))
print('Columnas canónicas:', len(silver_df.columns))
silver_df.select('service', 'pickup_datetime', 'pickup_location_id',
                 'total_amount', 'dq_valid').show(5, truncate=False)

## Comprobaciones

In [ ]:
# Las 163 entradas reconciliadas prueban que se procesó el corpus publicado, no una muestra.
assert check['passed'] and d['files_processed'] == 163
assert d['historical_files_reconciled'] == 144
print('Silver reconciliado al 100 % de los archivos disponibles.')
spark.stop()

## Siguiente paso

Gold agrega Silver en granos útiles para decisiones y forma una constelación de hechos
compartiendo dimensiones conformadas.